# Agent Loop：从任务分解到可恢复执行

本章用一个多文件订单数据修复任务，拆解单 Agent 如何制定计划、执行动作、读取环境反馈、修改计划并停止。


## 1 结论与学习目标

**Planning 适合步骤数量无法预先写死、步骤之间存在依赖、执行结果会改变后续动作的任务。**

生产 Agent 需要具备以下控制能力：

1. 把目标转换为带依赖关系的步骤。
2. 每次只执行当前可运行的步骤。
3. 把工具结果保存为观察证据。
4. 根据证据重试、重规划、请求人工处理或停止。
5. 使用校验器判断目标是否完成。
6. 使用预算、检查点和幂等机制限制失控范围。

本章最重要的工程分工如下：

| 部分 | 主要职责 |
| --- | --- |
| 模型 | 提议计划、选择下一动作、根据新证据修订计划 |
| 控制器 | 校验计划、调度步骤、执行工具、记录状态、实施预算和停止条件 |
| 工具 | 对外部环境执行确定性动作 |
| 校验器 | 使用可复现规则判断产物是否达标 |

简单查询继续使用单次模型调用或一次 Tool Calling。Planning 会增加模型调用、延迟、状态和失败分支，任务复杂度足够高时再使用。[4]


### 1.1 本章边界

本章只讨论**单 Agent Planning**。Planner、Controller、Verifier 是同一个 Agent 内部的职责模块，不代表多个 Agent。

第 4 章解决一次工具调用，第 7 章解决上下文、记忆和检查点，第 8 章把这些能力放进持续运行的决策循环。框架封装、多 Agent 协作、完整 Harness 和生产评测分别留到后续章节。


## 2 背景与原因

### 2.1 一次调用为什么不够

一次模型调用适合输入完整、动作明确、结果可直接返回的任务。以下任务需要循环执行：

- 开始时只知道目标，无法确定完整步骤。
- 后一步依赖前一步的真实结果。
- 工具可能失败，失败原因决定恢复动作。
- 需要修改多个外部对象，并在结束前统一验证。
- 运行时间较长，进程可能中断。

ReAct 将推理与动作交错执行，使模型能够根据环境反馈更新计划并处理异常。[1] Plan-and-Solve 先生成总体计划，再逐项执行，用于减少遗漏步骤。[2] 两者可以组合：先建立粗粒度计划，执行时继续观察和修订。


### 2.2 ReAct：最小的观察反馈循环

**ReAct 的主体放在 Agent Loop 章节。** ReAct 论文使用 Few-shot Prompt 展示 `Thought → Action → Observation` 轨迹，因此提示词工程可以介绍它的表达格式；完整机制还需要工具执行器、环境反馈、状态装配和停止判断，这些属于 Agent 运行循环。[1]

ReAct 每轮处理三个对象：

| 阶段 | 当前参与者 | 产物 | 工程实现 |
| --- | --- | --- | --- |
| Reason | 模型 | 下一动作及简短依据 | 根据目标、计划和最新观察选择动作 |
| Action | Controller + Tool | 结构化工具调用 | 校验权限和参数后执行 |
| Observation | Tool + Environment | 外部事实 | 把结果、错误码和证据写入状态 |

```mermaid
flowchart LR
    G["Goal + 当前状态"] --> R["Reason：选择下一动作"]
    R --> A["Action：执行工具"]
    A --> O["Observation：记录环境结果"]
    O --> V{"目标完成或必须停止"}
    V -->|"继续"| R
    V -->|"结束"| E["Final / Failed / Human"]
```

生产系统无需保存模型的完整私有思维链。Controller 保存简短决策依据、结构化 Action 和可验证 Observation，即可支持恢复、审计和评测。

ReAct 解决**下一步怎么根据新观察调整**；Planning 解决**全局有哪些步骤和依赖**。本章采用组合方式：先生成粗粒度 Plan，再用 ReAct 式循环逐步执行；观察使原计划失效时提交 Plan Patch。


### 2.3 Agent Loop 与 Planning 的关系

| 概念 | 回答的问题 | 典型产物 |
| --- | --- | --- |
| Tool Calling | 这一步调用哪个工具，参数是什么 | Tool Call |
| ReAct | 如何根据一次 Action 的 Observation 决定下一步 | Reason、Action、Observation |
| Planning | 为完成目标，需要哪些步骤，依赖和顺序是什么 | Plan |
| Agent Loop | 下一轮何时调用模型或工具，何时停止 | 状态迁移 |
| Context / Memory | 本轮决策需要带上哪些历史信息 | 上下文包、记忆、检查点 |

OpenAI 对 Codex agent loop 的描述是：模型返回最终消息或工具调用；应用执行工具并把结果追加到下一次模型输入；该过程持续到模型返回最终消息。[3] 生产实现还要补充目标验证、预算、错误分类和恢复逻辑。
思考：上下文压缩与可恢复之间的矛盾


### 2.4 什么时候使用哪种方式

| 任务特征 | 推荐方式 | 原因 |
| --- | --- | --- |
| 单一、确定、低风险 | 单次模型调用 | 成本最低 |
| 需要一个外部动作 | Tool Calling | 动作协议已经足够 |
| 步骤固定且业务规则稳定 | 代码工作流或状态机 | 路径可预测、便于审计 |
| 步骤会随环境反馈变化 | Agent Loop + Planning | 需要动态决策 |
| 写操作不可逆或风险高 | 固定工作流 + 人工审批 | 自治范围必须受控 |

工作流由代码预先规定路径，Agent 让模型动态决定过程。行业实践通常从可预测的工作流开始，只在固定路径无法覆盖时增加模型自主性。[4]


## 3 核心概念

### 3.1 统一术语

| 术语 | 定义 | 本章示例 |
| --- | --- | --- |
| Goal | 最终需要满足的结果 | 生成通过规则校验的订单文件和审计报告 |
| Constraint | 执行期间必须遵守的限制 | 金额大于 0、订单号全局唯一 |
| Plan | 带步骤、依赖和状态的执行方案 | 检查输入 → 清洗 → 校验 → 报告 |
| Step | 一个可独立调度和观察的动作 | `validate_output` |
| Dependency | 当前步骤运行前必须成功的步骤 | 报告依赖校验成功 |
| Observation | 工具执行后返回的事实 | `A102` 重复 |
| Evidence | 支撑步骤成功或失败的可保存证据 | 校验错误列表、产物路径、行数 |
| Replan | 根据新证据修改未完成计划 | 插入去重步骤和二次校验 |
| Budget | 一次运行允许消耗的上限 | 最多 8 次动作、2 次重规划 |
| Stop Condition | 终止循环的确定性条件 | 目标通过、预算耗尽、阻塞、人工接管 |
| Checkpoint | 可恢复的 Agent 状态快照 | 步骤状态、预算、观察和产物索引 |


### 3.2 计划保存什么

计划保存可执行信息：步骤目标、动作、依赖、状态、尝试次数、证据和错误码。私有推理过程无需写入计划，也不应成为控制器依赖的协议。

推荐只让模型返回简短依据，例如“校验发现重复订单，因此插入去重步骤”。控制器依赖结构化字段和环境证据完成调度。


## 4 贯穿案例：多文件订单数据修复 Agent

### 4.1 输入、目标与验收标准

Agent 收到两份 CSV 和一份规则文件。脏数据包含负金额、零金额、缺失币种和跨文件重复订单。

**目标产物：**

1. `orders_clean.csv`：保留合法订单，币种缺失时补为 `CNY`，订单号全局唯一。
2. `audit.json`：记录输入行数、删除原因、修复数量、输出行数和校验状态。

**完成条件：**

- `orders_clean.csv` 通过独立校验器。
- `audit.json` 存在，且 `validation=passed`。
- 控制器记录每一步的状态和证据。

该案例把“完成”转换为机器可验证条件，避免模型仅凭自然语言宣布成功。


### 4.2 参与者与职责

| 参与者 | 职责 |
| --- | --- |
| 用户 | 提供目标、输入和约束 |
| Planner | 把目标转换为计划；失败后提出计划变更 |
| Controller | 校验计划、选择可运行步骤、调用工具、实施预算与停止条件 |
| Tool | 读取、清洗、去重、校验和写报告 |
| Environment | 保存输入文件、候选产物和工具错误 |
| Verifier | 按规则检查产物，返回结构化证据 |
| State Store | 保存 Plan、Observation、Budget 和 Checkpoint |

Planner 和 Controller 可以调用同一个模型，也可以由“模型 + 普通代码”共同完成。它们都属于一个 Agent。


### 4.3 完整循环

```mermaid
flowchart TD
    G["Goal + Constraints"] --> P["生成或读取 Plan"]
    P --> V["校验计划结构与权限"]
    V --> S["选择依赖已满足的 Step"]
    S --> A["执行 Tool Action"]
    A --> O["记录 Observation + Evidence"]
    O --> Q{"目标是否通过 Verifier"}
    Q -->|"是"| D["Succeeded"]
    Q -->|"否"| E{"错误能否恢复"}
    E -->|"可以"| R["Retry 或 Replan"]
    R --> S
    E -->|"需要审批"| H["Needs Human"]
    E -->|"不可恢复或预算耗尽"| F["Failed / Blocked"]
```

一轮循环只做一个决策：选出下一步、执行、记录事实、重新判断。工具结果必须先进入状态，再允许 Planner 修改计划。


### 4.4 完整时序

```mermaid
sequenceDiagram
    participant U as User
    participant C as Controller
    participant P as Planner
    participant T as Tool
    participant V as Verifier
    participant S as State Store

    U->>C: Goal + Constraints
    C->>P: 生成初始计划
    P-->>C: Steps + Dependencies
    C->>S: 保存计划
    loop 直到终止
        C->>C: 选择可运行步骤并检查预算
        C->>T: 执行动作
        T-->>C: Tool Result
        C->>S: 保存 Observation 和证据
        C->>V: 校验当前产物
        V-->>C: 通过或结构化错误
        alt 需要调整
            C->>P: 当前计划 + 新证据
            P-->>C: Plan Patch
            C->>S: 保存新版本
        end
    end
    C-->>U: 状态 + 产物 + 证据
```


## 5 原理拆解

### 5.1 计划的数据结构

最小步骤协议包含以下字段：

```json
{
  "id": "validate_v1",
  "objective": "校验候选订单文件",
  "action": "validate_output",
  "depends_on": ["clean_v1"],
  "status": "pending",
  "attempts": 0,
  "evidence": null,
  "last_error": null
}
```

`objective` 给人和模型阅读，`action` 供控制器路由，`depends_on` 决定调度，剩余字段用于恢复和审计。只保存自然语言待办列表，控制器无法可靠判断依赖、重试和完成状态。


### 5.2 依赖关系与可运行步骤

计划通常表示为有向无环图（DAG）。步骤满足以下条件时才可运行：

1. 当前状态为 `pending`。
2. 所有 `depends_on` 步骤均为 `succeeded`。
3. 动作在白名单中，参数通过校验。
4. 预算未耗尽，权限策略允许执行。

```mermaid
flowchart LR
    I["inspect"] --> C["clean"]
    C --> V1["validate_v1"]
    V1 --> A["write_audit"]
    C -. "发现重复后插入" .-> D["deduplicate"]
    D --> V2["validate_v2"]
    V2 --> A
```

重规划保留已经成功的步骤和证据，只修改尚未完成的部分。这样可以减少重复执行和副作用。


### 5.3 重试与重规划

| 观察结果 | 处理方式 | 示例 |
| --- | --- | --- |
| 临时错误，原动作仍正确 | Retry | 网络超时、限流 |
| 输入或前提改变，原步骤已不适用 | Replan | 校验发现跨文件重复 |
| 缺少权限或需要业务判断 | Needs Human | 发布、删除、付款 |
| 参数或计划结构非法 | Reject Plan | 动作不在白名单、循环依赖 |
| 同类失败重复出现 | Blocked | 两次修复后仍是同一错误 |

Retry 必须有次数上限和退避策略。Replan 必须引用最新 Observation，并限制重规划次数。没有错误分类时，Agent 容易把确定性错误重复执行到预算耗尽。


### 5.4 停止条件

推荐按以下顺序判断：

1. **目标通过独立校验：** `succeeded`。
2. **需要不可逆动作或权限：** `needs_human`。
3. **没有可运行步骤：** `blocked`。
4. **动作、时间、Token 或重规划预算耗尽：** `budget_exhausted`。
5. **仍有可运行步骤：** 进入下一轮。

“模型返回最终回答”只能表示模型希望结束。控制器仍需检查目标、产物和预算。


### 5.5 Checkpoint 与恢复语义

Checkpoint 至少保存：计划版本、步骤状态、Observation、预算消耗、产物索引和等待原因。恢复时加载同一个运行 ID，从最后一个已确认检查点继续。

LangGraph 的持久化层按执行步骤保存 checkpoint，用于故障恢复、人工介入和状态回放。[5] 其中断机制恢复时会从节点开头重新执行，因此中断前的副作用需要幂等，或移动到中断之后。[6]

生产系统必须明确两种崩溃窗口：

- 工具尚未执行，checkpoint 已保存：恢复后可以正常执行。
- 工具已经成功，checkpoint 尚未保存：恢复后可能重复执行，需要幂等键、结果查询或事务性 outbox。


## 6 渐进代码实验

### 6.1 实验说明

本章固定工具实现，只改变控制方式。这样可以把“规划机制的效果”和“模型生成代码的质量”分开观察。

离线实验使用确定性 Planner 回放，Notebook 默认不会调用 API。最后提供 DeepSeek/OpenAI-compatible 真实 Planner 入口，设置 `RUN_LIVE_LLM=1` 后才会执行。


### 6.2 准备实验目录与配置

下面的代码创建临时目录。所有写操作都限制在该目录内，重复运行会重建数据。

配置加载方式与第 3、4 章一致：`find_dotenv(usecwd=True)` 从当前目录向上定位仓库根目录 `.env`，`load_dotenv(..., override=False)` 加载配置。Shell 或 Jupyter 进程已经存在的环境变量优先。


In [1]:
from __future__ import annotations

import csv
import json
import os
import shutil
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

from dotenv import find_dotenv, load_dotenv

# 从 Notebook 启动目录向上查找仓库根目录 .env。
dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)

RUN_LIVE_LLM = os.getenv("RUN_LIVE_LLM", "0") == "1"
LLM_API_KEY = (
    os.getenv("LLM_API_KEY")
    or os.getenv("DEEPSEEK_API_KEY")
    or os.getenv("DASHSCOPE_API_KEY")
    or os.getenv("OPENAI_API_KEY")
)
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "").strip()
LLM_MODEL = os.getenv("LLM_MODEL", "").strip()
DEMO_ROOT = Path(tempfile.mkdtemp(prefix="agent-planning-"))

safe_config = {
    "dotenv_found": bool(dotenv_path),
    "run_live_llm": RUN_LIVE_LLM,
    "api_key_configured": bool(LLM_API_KEY),
    "base_url": LLM_BASE_URL or "OpenAI default",
    "model": LLM_MODEL or "not configured",
}
print(safe_config)
assert "api_key" not in safe_config
assert DEMO_ROOT.exists()


{'dotenv_found': True, 'run_live_llm': False, 'api_key_configured': True, 'base_url': 'https://api.deepseek.com', 'model': 'deepseek-v4-pro'}


### 6.3 构造脏数据

两个输入文件共 6 行。`A101` 和 `A104` 金额非法，`A102` 跨文件重复，其中一行缺失币种。


In [2]:
FIELDNAMES = ["order_id", "city", "amount", "currency"]

INPUT_ROWS = {
    "orders_a.csv": [
        {"order_id": "A100", "city": "深圳", "amount": "100", "currency": "CNY"},
        {"order_id": "A101", "city": "上海", "amount": "-20", "currency": "CNY"},
        {"order_id": "A102", "city": "北京", "amount": "88", "currency": ""},
    ],
    "orders_b.csv": [
        {"order_id": "A102", "city": "北京", "amount": "88", "currency": "CNY"},
        {"order_id": "A103", "city": "杭州", "amount": "120", "currency": "CNY"},
        {"order_id": "A104", "city": "成都", "amount": "0", "currency": "CNY"},
    ],
}

RULES = {
    "required_columns": FIELDNAMES,
    "unique_key": "order_id",
    "default_currency": "CNY",
    "amount_must_be_greater_than": 0,
}


def check_rule(row: dict[str, str], rule_name: str) -> bool:
    # 规则名已经表达比较语义，这里只分发本案例实际使用的规则。
    match rule_name:
        case "amount_must_be_greater_than":
            return float(row["amount"]) > float(RULES[rule_name])
        case _:
            raise ValueError(f"不支持的规则: {rule_name}")


def write_rows(path: Path, rows: list[dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(rows)


def read_rows(path: Path) -> list[dict[str, str]]:
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def reset_workspace(path: Path) -> Path:
    if path.exists():
        shutil.rmtree(path)
    (path / "inputs").mkdir(parents=True)
    for filename, rows in INPUT_ROWS.items():
        write_rows(path / "inputs" / filename, rows)
    (path / "rules.json").write_text(
        json.dumps(RULES, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return path


WORKSPACE = reset_workspace(DEMO_ROOT / "main")
print({name: len(rows) for name, rows in INPUT_ROWS.items()})
assert sum(map(len, INPUT_ROWS.values())) == 6


{'orders_a.csv': 3, 'orders_b.csv': 3}


### 6.4 无计划执行：做了动作，没有完成目标

最直接的实现是把文件拼接后宣布完成。独立校验器会检查金额、币种和全局唯一性。


In [3]:
def validate_candidate(root: Path) -> list[dict[str, Any]]:
    candidate = root / "orders_clean.csv"
    if not candidate.exists():
        return [{"code": "missing_output", "detail": str(candidate)}]

    rows = read_rows(candidate)
    errors: list[dict[str, Any]] = []
    invalid_amounts = [
        row["order_id"]
        for row in rows
        if not check_rule(row, "amount_must_be_greater_than")
    ]
    missing_currency = [row["order_id"] for row in rows if not row["currency"]]

    seen: set[str] = set()
    duplicates: list[str] = []
    for row in rows:
        order_id = row["order_id"]
        if order_id in seen and order_id not in duplicates:
            duplicates.append(order_id)
        seen.add(order_id)

    if invalid_amounts:
        errors.append({"code": "non_positive_amount", "order_ids": invalid_amounts})
    if missing_currency:
        errors.append({"code": "missing_currency", "order_ids": missing_currency})
    if duplicates:
        errors.append({"code": "duplicate_order_id", "order_ids": duplicates})
    return errors


# 无计划路径只做“看起来最直接”的拼接。
naive_rows = [row for rows in INPUT_ROWS.values() for row in rows]
write_rows(WORKSPACE / "orders_clean.csv", naive_rows)
baseline_errors = validate_candidate(WORKSPACE)

print(json.dumps(baseline_errors, ensure_ascii=False, indent=2))
assert {error["code"] for error in baseline_errors} == {
    "non_positive_amount", "missing_currency", "duplicate_order_id"
}


[
  {
    "code": "non_positive_amount",
    "order_ids": [
      "A101",
      "A104"
    ]
  },
  {
    "code": "missing_currency",
    "order_ids": [
      "A102"
    ]
  },
  {
    "code": "duplicate_order_id",
    "order_ids": [
      "A102"
    ]
  }
]


校验结果证明了两件事：执行过工具不等于完成目标；最终状态需要由外部规则判断。下一步把目标转换为可调度的计划。


### 6.5 定义 Plan、Budget 与 AgentState

`PlanStep` 保存单步状态，`Budget` 保存硬上限，`AgentState` 保存完整运行事实。代码中的 `field(default_factory=...)` 表示每个对象分别创建自己的列表或字典，避免多个实例共享可变数据。


In [4]:
ALLOWED_ACTIONS = {
    "inspect_inputs",
    "clean_rows",
    "validate_output",
    "deduplicate_output",
    "write_audit",
}


@dataclass
class PlanStep:
    id: str
    objective: str
    action: str
    depends_on: list[str] = field(default_factory=list)
    status: str = "pending"
    attempts: int = 0
    evidence: dict[str, Any] | None = None
    last_error: dict[str, Any] | None = None


@dataclass
class Budget:
    max_actions: int = 8
    max_replans: int = 2


@dataclass
class AgentState:
    run_id: str
    goal: str
    steps: list[PlanStep]
    budget: Budget = field(default_factory=Budget)
    status: str = "running"
    actions_used: int = 0
    replans_used: int = 0
    observations: list[dict[str, Any]] = field(default_factory=list)
    artifacts: dict[str, Any] = field(default_factory=dict)


### 6.6 建立并校验初始计划

为了稳定复现实验，初始计划故意遗漏跨文件去重。Planner 可以犯错；系统必须通过执行证据发现并修复错误。

计划校验在执行前完成三项检查：ID 唯一、动作在白名单、依赖存在且无环。


In [5]:
def build_initial_state(max_actions: int = 8) -> AgentState:
    return AgentState(
        run_id="order-repair-001",
        goal="生成通过规则校验的订单文件和审计报告",
        budget=Budget(max_actions=max_actions, max_replans=2),
        steps=[
            PlanStep("inspect", "读取规则并统计输入", "inspect_inputs"),
            PlanStep("clean_v1", "补全币种并过滤非法金额", "clean_rows", ["inspect"]),
            PlanStep("validate_v1", "校验候选文件", "validate_output", ["clean_v1"]),
            PlanStep("audit", "写入审计报告", "write_audit", ["validate_v1"]),
        ],
    )


def validate_plan(steps: list[PlanStep]) -> None:
    ids = [step.id for step in steps]
    assert len(ids) == len(set(ids)), "步骤 ID 必须唯一"
    assert all(step.action in ALLOWED_ACTIONS for step in steps), "存在未授权动作"

    known = set(ids)
    assert all(set(step.depends_on) <= known for step in steps), "依赖步骤不存在"

    # 逐步移除依赖已满足的节点；无法全部移除时，计划中存在环。
    completed: set[str] = set()
    remaining = {step.id: set(step.depends_on) for step in steps}
    while remaining:
        ready = [step_id for step_id, deps in remaining.items() if deps <= completed]
        assert ready, "计划存在循环依赖"
        for step_id in ready:
            completed.add(step_id)
            remaining.pop(step_id)


state = build_initial_state()
validate_plan(state.steps)
print([(step.id, step.action, step.depends_on) for step in state.steps])


[('inspect', 'inspect_inputs', []), ('clean_v1', 'clean_rows', ['inspect']), ('validate_v1', 'validate_output', ['clean_v1']), ('audit', 'write_audit', ['validate_v1'])]


### 6.7 实现确定性工具

五个动作构成工具白名单。工具返回统一的 `ActionResult`，成功和失败都带结构化证据。Planner 只能选择这些动作，不能直接执行任意代码。

规则名已经包含判断语义。`check_rule(row, rule_name)` 使用 `match/case` 处理当前案例实际存在的金额规则，清洗器只消费布尔结果。


In [6]:
@dataclass
class ActionResult:
    ok: bool
    code: str
    detail: dict[str, Any]


def inspect_inputs(root: Path, state: AgentState) -> ActionResult:
    files = sorted((root / "inputs").glob("*.csv"))
    total_rows = sum(len(read_rows(path)) for path in files)
    state.artifacts["input_rows"] = total_rows
    return ActionResult(True, "inputs_inspected", {
        "files": [path.name for path in files], "input_rows": total_rows
    })


def clean_rows(root: Path, state: AgentState) -> ActionResult:
    source_rows = [
        row
        for path in sorted((root / "inputs").glob("*.csv"))
        for row in read_rows(path)
    ]
    cleaned: list[dict[str, str]] = []
    normalized_currency = 0
    filtered_invalid_amount = 0

    for row in source_rows:
        if not check_rule(row, "amount_must_be_greater_than"):
            filtered_invalid_amount += 1
            continue
        row = dict(row)  # 复制后修改，保留原始输入。
        if not row["currency"]:
            row["currency"] = RULES["default_currency"]
            normalized_currency += 1
        cleaned.append(row)

    write_rows(root / "orders_clean.csv", cleaned)
    state.artifacts.update({
        "normalized_currency": normalized_currency,
        "filtered_invalid_amount": filtered_invalid_amount,
    })
    return ActionResult(True, "rows_cleaned", {"candidate_rows": len(cleaned)})


def validate_output(root: Path, state: AgentState) -> ActionResult:
    errors = validate_candidate(root)
    if errors:
        return ActionResult(False, errors[0]["code"], {"errors": errors})
    output_rows = len(read_rows(root / "orders_clean.csv"))
    state.artifacts.update({"validation": "passed", "output_rows": output_rows})
    return ActionResult(True, "validation_passed", {"output_rows": output_rows})


In [7]:
def deduplicate_output(root: Path, state: AgentState) -> ActionResult:
    rows = read_rows(root / "orders_clean.csv")
    seen: set[str] = set()
    unique_rows: list[dict[str, str]] = []
    removed = 0

    for row in rows:
        if row["order_id"] in seen:
            removed += 1
            continue
        seen.add(row["order_id"])
        unique_rows.append(row)

    # 覆盖写入使动作具备幂等性：重复运行不会继续删除合法数据。
    write_rows(root / "orders_clean.csv", unique_rows)
    state.artifacts["removed_duplicates"] = removed
    return ActionResult(True, "duplicates_removed", {"removed": removed})


def write_audit(root: Path, state: AgentState) -> ActionResult:
    audit = {
        "input_rows": state.artifacts.get("input_rows", 0),
        "normalized_currency": state.artifacts.get("normalized_currency", 0),
        "filtered_invalid_amount": state.artifacts.get("filtered_invalid_amount", 0),
        "removed_duplicates": state.artifacts.get("removed_duplicates", 0),
        "output_rows": state.artifacts.get("output_rows", 0),
        "validation": state.artifacts.get("validation", "unknown"),
    }
    (root / "audit.json").write_text(
        json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return ActionResult(True, "audit_written", {"path": "audit.json"})


ACTION_HANDLERS = {
    "inspect_inputs": inspect_inputs,
    "clean_rows": clean_rows,
    "validate_output": validate_output,
    "deduplicate_output": deduplicate_output,
    "write_audit": write_audit,
}

assert set(ACTION_HANDLERS) == ALLOWED_ACTIONS


### 6.8 选择步骤、执行动作、记录观察

Controller 只选择依赖均成功的 `pending` 步骤。执行后立即更新尝试次数、预算、状态、证据和错误。


In [8]:
def next_runnable_step(state: AgentState) -> PlanStep | None:
    succeeded = {step.id for step in state.steps if step.status == "succeeded"}
    for step in state.steps:
        if step.status == "pending" and set(step.depends_on) <= succeeded:
            return step
    return None


def execute_step(root: Path, state: AgentState, step: PlanStep) -> ActionResult:
    step.status = "running"
    step.attempts += 1
    state.actions_used += 1

    result = ACTION_HANDLERS[step.action](root, state)
    observation = {
        "step_id": step.id,
        "action": step.action,
        "ok": result.ok,
        "code": result.code,
        "detail": result.detail,
    }
    state.observations.append(observation)

    if result.ok:
        step.status = "succeeded"
        step.evidence = result.detail
        step.last_error = None
    else:
        step.status = "failed"
        step.last_error = {"code": result.code, "detail": result.detail}
    return result


test_state = build_initial_state()
first = next_runnable_step(test_state)
assert first is not None and first.id == "inspect"


### 6.9 根据失败证据重规划

初始清洗会修复币种并过滤非法金额，但仍保留跨文件重复订单。校验返回 `duplicate_order_id` 后，Replanner 执行以下 Plan Patch：

1. 保留已经成功的 `inspect` 和 `clean_v1`。
2. 插入 `deduplicate`。
3. 插入 `validate_v2`。
4. 把审计步骤的依赖从失败的 `validate_v1` 改为 `validate_v2`。


In [9]:
def replan_after_failure(state: AgentState, failed_step: PlanStep) -> bool:
    # 返回 True 表示计划已经修改；返回 False 表示当前错误无法在这里处理。

    # failed_step.last_error 可能是 None，因此先用空字典兜底，再读取错误码。
    error_code = (failed_step.last_error or {}).get("code")

    # 本实验只演示一种重规划：发现重复订单后，插入去重和二次校验。
    # 其他错误交给 Controller 选择重试、人工介入或停止。
    if error_code != "duplicate_order_id":
        return False

    # 重规划也要消耗预算。达到上限后禁止继续修改计划，避免无限循环。
    if state.replans_used >= state.budget.max_replans:
        return False

    # next(...) 从 state.steps 中找到第一条满足条件的步骤。
    # clean_rows 已经成功，它的产物可以复用；新去重步骤应依赖它。
    clean_step = next(
        step for step in state.steps
        if step.action == "clean_rows" and step.status == "succeeded"
    )

    # 找到尚未执行的审计步骤。首次校验失败后，它原来的依赖已经失效，
    # 后面需要把依赖改成新的二次校验步骤。
    audit_step = next(
        step for step in state.steps
        if step.action == "write_audit" and step.status == "pending"
    )

    # 为新步骤生成唯一 ID。第一次重规划会得到 deduplicate_r1 和 validate_r2。
    suffix = state.replans_used + 1
    dedup_id = f"deduplicate_r{suffix}"
    validate_id = f"validate_r{suffix + 1}"

    # 在原计划末尾追加两个步骤：先对 clean_rows 的产物去重，再重新校验。
    # PlanStep 最后一个参数是依赖列表：依赖步骤成功后，当前步骤才能运行。
    state.steps.extend([
        PlanStep(dedup_id, "删除跨文件重复订单", "deduplicate_output", [clean_step.id]),
        PlanStep(validate_id, "重新校验修复后的文件", "validate_output", [dedup_id]),
    ])

    # 原路径：validate_v1 -> audit。
    # 新路径：deduplicate_r1 -> validate_r2 -> audit。
    audit_step.depends_on = [validate_id]

    # 更新预算计数，并记录这次 Plan Patch，供日志、恢复和调试使用。
    state.replans_used += 1
    state.observations.append({
        "type": "plan_patch",
        "reason": error_code,
        "added": [dedup_id, validate_id],
        "updated": {audit_step.id: audit_step.depends_on},
    })

    # 修改完成后重新检查 ID、动作白名单、依赖是否存在以及是否有环。
    validate_plan(state.steps)
    return True


### 6.10 保存和加载 Checkpoint

Checkpoint 使用 JSON 保存，便于观察结构。生产系统通常使用数据库，并通过 `run_id`、版本号和事务控制并发恢复。


In [10]:
def save_checkpoint(state: AgentState, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(asdict(state), ensure_ascii=False, indent=2), encoding="utf-8"
    )


def load_checkpoint(path: Path) -> AgentState:
    payload = json.loads(path.read_text(encoding="utf-8"))
    payload["steps"] = [PlanStep(**item) for item in payload["steps"]]
    payload["budget"] = Budget(**payload["budget"])
    return AgentState(**payload)


checkpoint_test = DEMO_ROOT / "checkpoint-test.json"
save_checkpoint(build_initial_state(), checkpoint_test)
restored = load_checkpoint(checkpoint_test)
assert restored.run_id == "order-repair-001"
assert len(restored.steps) == 4


### 6.11 组装最小 Agent Loop

循环按固定顺序处理：验证目标、检查预算、选择步骤、执行、观察、重规划、保存检查点。`pause_after` 只用于模拟进程中断。

这段循环同时实现了 ReAct：

1. **Reason**：`next_runnable_step` 根据 Plan、依赖和状态选择下一动作；失败后 `replan_after_failure` 根据 Observation 修订路径。
2. **Action**：`execute_step` 从白名单路由并执行工具。
3. **Observation**：`execute_step` 把 `ok`、错误码和证据写入 `state.observations`。

离线实验的 Reason 由确定性调度器完成。真实 Agent 可以让模型提出下一动作或 Plan Patch，Controller 继续负责校验、执行和停止。


In [11]:
TERMINAL_STATES = {"succeeded", "blocked", "budget_exhausted", "needs_human"}


def goal_verified(root: Path, state: AgentState) -> bool:
    audit_path = root / "audit.json"
    if not audit_path.exists() or validate_candidate(root):
        return False
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    return audit.get("validation") == "passed"


def run_loop(
    root: Path,
    state: AgentState,
    checkpoint_path: Path,
    pause_after: int | None = None,
) -> AgentState:
    if state.status == "paused":
        state.status = "running"
    actions_in_this_process = 0

    while state.status not in TERMINAL_STATES:
        if goal_verified(root, state):
            state.status = "succeeded"
            break
        if state.actions_used >= state.budget.max_actions:
            state.status = "budget_exhausted"
            break

        # Reason：根据计划依赖和当前状态选择下一步，不在这里执行工具。
        step = next_runnable_step(state)
        if step is None:
            state.status = "blocked"
            break

        # Action + Observation：执行白名单动作，并把结构化结果写入 state.observations。
        result = execute_step(root, state, step)
        actions_in_this_process += 1
        if not result.ok:
            print(step, result)
            # 新 Observation 证明原路径失效，再进入下一轮 Reason 并提交 Plan Patch。
            replan_after_failure(state, step)

        save_checkpoint(state, checkpoint_path)
        if pause_after and actions_in_this_process >= pause_after:
            state.status = "paused"
            save_checkpoint(state, checkpoint_path)
            return state

    save_checkpoint(state, checkpoint_path)
    return state


## 7 完整实验

### 7.1 执行、失败、重规划、成功

预期轨迹共 6 次动作：检查输入、初步清洗、首次校验失败、去重、二次校验、写审计报告。


下图中的珊瑚红节点是首次失败位置。`validate_v1` 返回 `duplicate_order_id: A102` 后，Controller 保留已经成功的 `inspect` 和 `clean_v1`，把执行路径下跳到 `Plan Patch`，再从新插入的 `deduplicate_r1` 继续。原来指向 `write_audit` 的依赖被切断，二次校验通过后重新接通。

![首次校验失败后的重规划与执行路径](imgs/04-framework-replan-routing-v4.png)


In [12]:
WORKSPACE = reset_workspace(DEMO_ROOT / "complete-run")
state = build_initial_state(max_actions=8)
final_state = run_loop(
    WORKSPACE,
    state,
    WORKSPACE / "checkpoints" / "latest.json",
)

print("status:", final_state.status)
for item in final_state.observations:
    print(json.dumps(item, ensure_ascii=False))

clean_rows_result = read_rows(WORKSPACE / "orders_clean.csv")
audit_result = json.loads((WORKSPACE / "audit.json").read_text(encoding="utf-8"))

assert final_state.status == "succeeded"
assert final_state.actions_used == 6
assert final_state.replans_used == 1
assert [row["order_id"] for row in clean_rows_result] == ["A100", "A102", "A103"]
assert audit_result == {
    "input_rows": 6,
    "normalized_currency": 1,
    "filtered_invalid_amount": 2,
    "removed_duplicates": 1,
    "output_rows": 3,
    "validation": "passed",
}


PlanStep(id='validate_v1', objective='校验候选文件', action='validate_output', depends_on=['clean_v1'], status='failed', attempts=1, evidence=None, last_error={'code': 'duplicate_order_id', 'detail': {'errors': [{'code': 'duplicate_order_id', 'order_ids': ['A102']}]}}) ActionResult(ok=False, code='duplicate_order_id', detail={'errors': [{'code': 'duplicate_order_id', 'order_ids': ['A102']}]})
status: succeeded
{"step_id": "inspect", "action": "inspect_inputs", "ok": true, "code": "inputs_inspected", "detail": {"files": ["orders_a.csv", "orders_b.csv"], "input_rows": 6}}
{"step_id": "clean_v1", "action": "clean_rows", "ok": true, "code": "rows_cleaned", "detail": {"candidate_rows": 4}}
{"step_id": "validate_v1", "action": "validate_output", "ok": false, "code": "duplicate_order_id", "detail": {"errors": [{"code": "duplicate_order_id", "order_ids": ["A102"]}]}}
{"type": "plan_patch", "reason": "duplicate_order_id", "added": ["deduplicate_r1", "validate_r2"], "updated": {"audit": ["validate_r2"

首次校验失败属于正常控制流。失败提供了可操作证据，Replanner 只修改未完成计划，二次校验通过后才允许写审计报告和结束任务。


### 7.2 从执行轨迹观察 ReAct

`final_state.observations` 已经保存每次 Action 的结果和重规划记录。下面把它转换为 ReAct 视图：`reason` 只写可公开的决策依据，`action` 保存结构化动作，`observation` 保存环境事实。

这个视图证明 ReAct 无需单独再造一套 Agent。它是现有循环的决策节奏；Plan、Budget、Checkpoint 和 Verifier 在这个节奏外增加生产控制。


In [13]:
def build_react_trace(state: AgentState) -> list[dict[str, Any]]:
    """把已保存的执行事实整理为 ReAct 视图，不记录模型私有思维链。"""

    steps_by_id = {step.id: step for step in state.steps}
    trace: list[dict[str, Any]] = []

    for observation in state.observations:
        if observation.get("type") == "plan_patch":
            trace.append({
                "reason": f"观察到 {observation['reason']}，原计划路径需要修改",
                "action": "plan.patch",
                "observation": {
                    "added": observation["added"],
                    "updated": observation["updated"],
                },
            })
            continue

        step = steps_by_id[observation["step_id"]]
        trace.append({
            "reason": f"依赖已满足，执行步骤 {step.id}：{step.objective}",
            "action": observation["action"],
            "observation": {
                "ok": observation["ok"],
                "code": observation["code"],
                "detail": observation["detail"],
            },
        })

    return trace


react_trace = build_react_trace(final_state)
for round_no, item in enumerate(react_trace, start=1):
    print(f"Round {round_no}")
    print("  Reason:", item["reason"])
    print("  Action:", item["action"])
    print("  Observation:", json.dumps(item["observation"], ensure_ascii=False))

assert sum(item["action"] != "plan.patch" for item in react_trace) == 6
assert any(item["action"] == "plan.patch" for item in react_trace)
assert any(item["observation"].get("code") == "duplicate_order_id" for item in react_trace)


Round 1
  Reason: 依赖已满足，执行步骤 inspect：读取规则并统计输入
  Action: inspect_inputs
  Observation: {"ok": true, "code": "inputs_inspected", "detail": {"files": ["orders_a.csv", "orders_b.csv"], "input_rows": 6}}
Round 2
  Reason: 依赖已满足，执行步骤 clean_v1：补全币种并过滤非法金额
  Action: clean_rows
  Observation: {"ok": true, "code": "rows_cleaned", "detail": {"candidate_rows": 4}}
Round 3
  Reason: 依赖已满足，执行步骤 validate_v1：校验候选文件
  Action: validate_output
  Observation: {"ok": false, "code": "duplicate_order_id", "detail": {"errors": [{"code": "duplicate_order_id", "order_ids": ["A102"]}]}}
Round 4
  Reason: 观察到 duplicate_order_id，原计划路径需要修改
  Action: plan.patch
  Observation: {"added": ["deduplicate_r1", "validate_r2"], "updated": {"audit": ["validate_r2"]}}
Round 5
  Reason: 依赖已满足，执行步骤 deduplicate_r1：删除跨文件重复订单
  Action: deduplicate_output
  Observation: {"ok": true, "code": "duplicates_removed", "detail": {"removed": 1}}
Round 6
  Reason: 依赖已满足，执行步骤 validate_r2：重新校验修复后的文件
  Action: validate_output
  Observation: {

### 7.3 预算耗尽：停止且不谎报成功

把动作预算限制为 2。Agent 完成检查和初步清洗后必须停止，校验尚未执行，因此状态只能是 `budget_exhausted`。


In [14]:
budget_root = reset_workspace(DEMO_ROOT / "budget-run")
budget_state = run_loop(
    budget_root,
    build_initial_state(max_actions=2),
    budget_root / "checkpoints" / "latest.json",
)

print({
    "status": budget_state.status,
    "actions_used": budget_state.actions_used,
    "audit_exists": (budget_root / "audit.json").exists(),
})

assert budget_state.status == "budget_exhausted"
assert budget_state.actions_used == 2
assert not (budget_root / "audit.json").exists()


{'status': 'budget_exhausted', 'actions_used': 2, 'audit_exists': False}


### 7.4 中断恢复：从已确认步骤继续

下面在两次动作后模拟中断。恢复进程读取 checkpoint，已经成功的 `inspect` 和 `clean_v1` 不再执行。


In [15]:
resume_root = reset_workspace(DEMO_ROOT / "resume-run")
resume_checkpoint = resume_root / "checkpoints" / "latest.json"

paused_state = run_loop(
    resume_root,
    build_initial_state(max_actions=8),
    resume_checkpoint,
    pause_after=2,
)
print("before restart:", paused_state.status, paused_state.actions_used)
assert paused_state.status == "paused"

restored_state = load_checkpoint(resume_checkpoint)
resumed_state = run_loop(resume_root, restored_state, resume_checkpoint)
print("after restart:", resumed_state.status, resumed_state.actions_used)

assert resumed_state.status == "succeeded"
assert resumed_state.actions_used == 6
assert [step.attempts for step in resumed_state.steps if step.id in {"inspect", "clean_v1"}] == [1, 1]


before restart: paused 2
PlanStep(id='validate_v1', objective='校验候选文件', action='validate_output', depends_on=['clean_v1'], status='failed', attempts=1, evidence=None, last_error={'code': 'duplicate_order_id', 'detail': {'errors': [{'code': 'duplicate_order_id', 'order_ids': ['A102']}]}}) ActionResult(ok=False, code='duplicate_order_id', detail={'errors': [{'code': 'duplicate_order_id', 'order_ids': ['A102']}]})
after restart: succeeded 6


### 7.5 DeepSeek / OpenAI-compatible 真实 Planner

离线实验固定了错误计划，便于观察 Loop。真实实验让模型根据目标、规则和动作目录生成初始计划，再交给同一个本地校验器和 Controller。

`.env` 配置示例：

```dotenv
DEEPSEEK_API_KEY=你的密钥
LLM_BASE_URL=https://api.deepseek.com
LLM_MODEL=你账号可用的模型名
RUN_LIVE_LLM=1
```

Notebook 不打印 API Key。提交前把 `RUN_LIVE_LLM` 恢复为 `0`，并清除真实响应输出。


In [16]:
from openai import OpenAI


def parse_json_object(text: str) -> dict[str, Any]:
    # 兼容模型偶尔返回 Markdown 围栏：截取最外层 JSON 对象。
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("模型响应中没有 JSON 对象")
    return json.loads(text[start:end + 1])


def steps_from_payload(payload: dict[str, Any]) -> list[PlanStep]:
    steps = [
        PlanStep(
            id=item["id"],
            objective=item["objective"],
            action=item["action"],
            depends_on=item.get("depends_on", []),
        )
        for item in payload["steps"]
    ]
    validate_plan(steps)
    return steps


def request_live_plan() -> dict[str, Any]:
    if not LLM_API_KEY or not LLM_MODEL:
        raise RuntimeError("需要配置 DEEPSEEK_API_KEY/LLM_API_KEY 和 LLM_MODEL")

    action_catalog = {
        "inspect_inputs": "读取规则、文件名和总行数",
        "clean_rows": "补全缺失币种并过滤金额小于等于 0 的记录",
        "validate_output": "检查金额、币种和订单号全局唯一性",
        "deduplicate_output": "按 order_id 保留第一条记录",
        "write_audit": "校验通过后写审计报告",
    }
    prompt = (
        "你是单 Agent 的 Planner。请为订单数据修复任务生成 DAG 计划。\n"
        "目标：生成通过规则校验的 orders_clean.csv 和 audit.json。\n"
        f"已知规则：{json.dumps(RULES, ensure_ascii=False)}\n"
        f"允许动作：{json.dumps(action_catalog, ensure_ascii=False)}\n"
        '只返回 JSON：{"steps":[{"id":"...","objective":"...",'
        '"action":"...","depends_on":[]}]}。\n'
        "每个动作最多使用一次；write_audit 必须依赖 validate_output。"
    )

    client_options = {"api_key": LLM_API_KEY}
    if LLM_BASE_URL:
        client_options["base_url"] = LLM_BASE_URL
    client = OpenAI(**client_options)
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return parse_json_object(response.choices[0].message.content or "")

load_dotenv(find_dotenv())
if RUN_LIVE_LLM:
    live_payload = request_live_plan()
    live_steps = steps_from_payload(live_payload)
    live_root = reset_workspace(DEMO_ROOT / "live-run")
    live_state = AgentState(
        run_id="order-repair-live",
        goal="生成通过规则校验的订单文件和审计报告",
        steps=live_steps,
    )
    live_result = run_loop(
        live_root, live_state, live_root / "checkpoints" / "latest.json"
    )
    print({"status": live_result.status, "steps": [asdict(s) for s in live_result.steps]})
else:
    print("live planner skipped: set RUN_LIVE_LLM=1 and configure API credentials")


live planner skipped: set RUN_LIVE_LLM=1 and configure API credentials


真实 Planner 可能一开始就加入去重，也可能遗漏某一步。结构校验只能保证计划格式可执行；业务完成度仍由 Verifier 判断。生产评测应记录初始计划通过率、重规划次数、最终成功率、动作数、Token、延迟和人工介入次数。


## 8 生产工程边界

### 8.1 计划粒度

| 粒度 | 优点 | 风险 | 适用情况 |
| --- | --- | --- | --- |
| 粗粒度 | 计划短、Token 少 | 单步失败后难定位和恢复 | 低风险读取任务 |
| 中粒度 | 证据、重试和依赖清晰 | 状态数量适中 | 默认选择 |
| 细粒度 | 控制和审计最强 | 调用次数、延迟、状态快速增加 | 高风险写操作 |

一个步骤应对应一个可观察结果或一个清晰副作用。连续读取同一对象并做紧密计算可以合并；权限不同、失败恢复不同或副作用不同的动作应拆开。


### 8.2 失败分类与恢复策略

| 失败类型 | 识别信号 | 默认处理 |
| --- | --- | --- |
| Transient | 超时、429、短暂 5xx | 有上限的指数退避重试 |
| Validation | Schema、业务规则、测试失败 | 记录证据并重规划 |
| Permission | 401、403、审批缺失 | 请求授权或人工接管 |
| Conflict | 版本号变化、并发写冲突 | 重新读取状态后重规划 |
| Irreversible | 删除、发布、转账 | 执行前人工审批 |
| Unknown | 未分类异常 | 停止并保留 checkpoint |

错误对象至少包含 `code`、`retryable`、`detail`、`step_id` 和 `timestamp`。模型阅读简短错误摘要，完整日志存入可按需读取的文件或 Trace。


### 8.3 幂等、重试与副作用

- 读取和覆盖式生成通常容易做到幂等。
- 创建、发送、追加、扣款和发布需要幂等键。
- 工具超时不代表工具未执行；重试前先查询结果。
- Step 状态和外部副作用无法放入同一事务时，使用 outbox、操作记录或补偿动作。
- 恢复后只调度 `pending` 步骤；`running` 步骤先核对外部执行结果。


### 8.4 人工介入点

以下情况进入 `needs_human`：

1. 目标或验收标准存在关键歧义。
2. 需要新增权限、访问敏感数据或突破原预算。
3. 下一步产生不可逆副作用。
4. 同类失败达到阈值。
5. Planner 提出的动作不在白名单。

人工界面应展示当前目标、已完成步骤、关键证据、待执行动作、影响范围和可选决策。审批结果作为新的 Observation 写入状态。


### 8.5 预算与停止条件

生产预算至少覆盖：

- 最大模型调用次数。
- 最大工具调用次数。
- 最大重试和重规划次数。
- 最大输入/输出 Token。
- 最大运行时间和单工具超时。
- 最大金额、数据行数或写操作数量等业务额度。

预算应由控制器计数，模型只能建议调整。扩大预算属于策略决策，达到阈值后自动继续会削弱成本和风险控制。


### 8.6 版本、并发与可观测性

- Plan 每次修改生成新版本，保留 Plan Patch 和修改原因。
- 同一 `run_id` 使用乐观锁，避免两个 Worker 同时推进同一步。
- Observation 保存摘要和原始证据位置，避免把大日志重复塞进上下文。
- Trace 关联 `run_id`、`plan_version`、`step_id`、模型调用和工具调用。
- 指标同时观察成功率、成本、延迟、重复动作、重规划和人工介入。

第 10 章会把这些能力放入完整 Harness，第 12 章继续实现 Evals、Tracing、Guardrails 和人审。


### 8.7 扩展阅读：其他 Agent 控制范式

以下方法分别优化计划生成、模型调用次数、跨尝试学习和搜索空间。它们可以组合，但每增加一种机制都会增加状态、调用和评测成本。

| 方法 | 核心做法 | 适合场景 | 主要代价 |
| --- | --- | --- | --- |
| Plan-and-Solve | 先生成总体计划，再按计划求解 | 容易漏步骤的多步任务 | 初始计划错误会向后传播；执行中仍需反馈修订 [2] |
| ReWOO | Planner 先生成带变量占位的工具计划，Worker 执行，Solver 汇总 | 工具路径较稳定、希望减少反复调用模型 | 计划阶段看不到工具结果，动态适应能力较弱 [7] |
| Reflexion | 根据失败反馈生成文字反思，写入情景记忆供下一次尝试使用 | 编码、调试和允许多次尝试的任务 | 增加尝试次数和记忆管理；错误反思会污染后续决策 [8] |
| Tree of Thoughts | 同时探索多条候选路径，评分、回溯和剪枝 | 组合搜索、需要前瞻的难题 | 模型调用量和搜索状态显著增加 [9] |

选择顺序：先建立单路径 ReAct 或固定工作流；明确发现“初始计划遗漏”“重复调用成本”“跨尝试不学习”或“单路径搜索失败”后，再引入对应方法。Self-Consistency 和 Tree of Thoughts 更接近推理时采样与搜索，第 3 章的提示词工程负责其基本概念，本章关注它们进入 Agent 运行时后的成本和控制。


## 9 常见踩坑

| 问题 | 直接后果 | 修复方式 |
| --- | --- | --- |
| Plan 只有自然语言列表 | 无法可靠调度和恢复 | 增加 ID、动作、依赖、状态和证据 |
| 每一步都重新生成完整计划 | 已完成步骤被改写，Token 增长 | 使用 Plan Patch，只改未完成部分 |
| 模型自行宣布成功 | 产物缺失或规则未通过 | 使用独立 Verifier |
| 所有错误统一重试 | 确定性错误持续消耗预算 | 错误分类后选择 retry/replan/human/stop |
| 恢复时直接重跑 `running` 步骤 | 重复发送或重复写入 | 查询外部结果并使用幂等键 |
| 工具返回整份日志 | 上下文快速膨胀 | 返回摘要、错误码和证据路径 |
| 动态计划可调用任意动作 | 权限边界失效 | 动作白名单、参数校验和审批策略 |
| 没有最大轮数 | Agent 循环无法收敛 | 动作、重试、重规划、时间和 Token 预算 |


## 10 上线检查清单

- [ ] Goal 和完成条件可机器验证。
- [ ] Step 具有唯一 ID、动作、依赖、状态和证据。
- [ ] Plan 在执行前检查白名单、依赖和循环。
- [ ] Retry 与 Replan 使用不同策略和预算。
- [ ] 每种终止状态都有明确语义。
- [ ] 不可逆动作执行前进入人工审批。
- [ ] 工具副作用支持幂等或补偿。
- [ ] Checkpoint 能恢复步骤、预算和产物索引。
- [ ] 同一运行只有一个推进者，或有并发版本控制。
- [ ] Trace 能串联计划版本、模型调用和工具调用。
- [ ] 评测同时覆盖成功、成本、延迟和失控路径。


## 11 本章小结

1. Agent Loop 持续连接模型、工具、环境反馈和终止判断。
2. Plan 是可执行状态协议，包含步骤、依赖、状态、证据和错误。
3. Controller 掌握动作白名单、预算、调度和停止权。
4. Verifier 使用环境事实判断完成，模型回答不能代替验收。
5. Retry 处理临时失败，Replan 处理路径失效，人工处理权限和高风险决策。
6. Checkpoint 解决进程恢复，幂等机制解决副作用重复。
7. 简单任务优先使用直接调用或固定工作流，复杂度达到需要动态反馈时再加入 Planning。


## 12 与最终项目的关系

产研协作助手会遇到“读取需求 → 查询任务 → 检查测试 → 汇总风险 → 等待审批 → 更新状态”这类多步骤任务。第 13 章将复用本章的 Plan、Observation、Budget、Verifier 和 Checkpoint 结构，并替换为真实业务工具与权限策略。

本章不提前实现该产品，只提供可复用的规划和执行控制原理。


## 13 下一步

第 9 章学习 Agent Skills 与 Agent-facing CLI：把可复用操作流程整理为 Skill，把原来面向人的 Web/API 能力改造成稳定、结构化、可授权、可审计的 Agent 执行接口。


## 参考文献

[1] YAO S, ZHAO J, YU D, et al. ReAct: Synergizing reasoning and acting in language models[C]//International Conference on Learning Representations. 2023. https://arxiv.org/abs/2210.03629.

[2] WANG L, XU W, LAN Y, et al. Plan-and-Solve Prompting: Improving zero-shot chain-of-thought reasoning by large language models[C]//Proceedings of the 61st Annual Meeting of the Association for Computational Linguistics. Toronto: Association for Computational Linguistics, 2023: 2609-2634. https://aclanthology.org/2023.acl-long.147/.

[3] OPENAI. Unrolling the Codex agent loop[EB/OL]. (2026-01-23)[2026-07-16]. https://openai.com/index/unrolling-the-codex-agent-loop/.

[4] ANTHROPIC. Building effective agents[EB/OL]. (2024-12-19)[2026-07-16]. https://www.anthropic.com/engineering/building-effective-agents.

[5] LANGCHAIN. Persistence[EB/OL]. [2026-07-16]. https://docs.langchain.com/oss/python/langgraph/persistence.

[6] LANGCHAIN. Interrupts[EB/OL]. [2026-07-16]. https://docs.langchain.com/oss/python/langgraph/interrupts.

[7] XU B, PENG Z, LEI B, et al. ReWOO: Decoupling reasoning from observations for efficient augmented language models[EB/OL]. (2023-05-30)[2026-07-16]. https://arxiv.org/abs/2305.18323.

[8] SHINN N, CASSANO F, GOPINATH A, et al. Reflexion: Language agents with verbal reinforcement learning[C]//Advances in Neural Information Processing Systems. 2023, 36: 8634-8652. https://papers.neurips.cc/paper_files/paper/2023/hash/1b44b878bb782e6954cd888628510e90-Abstract-Conference.html.

[9] YAO S, YU D, ZHAO J, et al. Tree of Thoughts: Deliberate problem solving with large language models[C]//Advances in Neural Information Processing Systems. 2023, 36: 11809-11822. https://proceedings.neurips.cc/paper_files/paper/2023/hash/271db9922b8d1f4dd7aaef84ed5ac703-Abstract.html.
